# 고객 데이터 정제 (Data Cleaning)

**데이터셋**: `messy_customer_data.csv`  
**정제 항목**:
1. customer_name → Title Case
2. Email 유효성 검사 + email_valid 플래그
3. Phone → (XXX) XXX-XXXX
4. signup_date → YYYY-MM-DD
5. Annual_Revenue → float
6. Country → 국가 전체 이름
7. Age → 정수, 불가능한 값 제거
8. is_active → True/False
9. 중복 행 제거
10. 전/후 비교 및 저장

In [3]:
import pandas as pd
import numpy as np
import re
import io
from dateutil import parser as dateparser
import warnings
warnings.filterwarnings('ignore')

# ── 불량 행(열 수 초과) 전처리 후 로드 ─────────────────────────────────────
with open('messy_customer_data.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()

header_cols = len(lines[0].split(','))
fixed_lines = [lines[0]]

for line in lines[1:]:
    if not line.strip():
        continue
    parts = line.rstrip('\n').split(',')
    # 열 수가 초과하면 앞쪽 빈 필드를 하나씩 제거
    while len(parts) > header_cols:
        for i in range(2, len(parts)):
            if parts[i] == '':
                parts.pop(i)
                break
    fixed_lines.append(','.join(parts) + '\n')

df = pd.read_csv(io.StringIO(''.join(fixed_lines)))

df_orig = df.copy()
print(f'로드 완료: {df.shape[0]}행 × {df.shape[1]}열')
df.head()

로드 완료: 100행 × 9열


,CustomerID,customer_name,Email,Phone,signup_date,Annual_Revenue,Country,Age,is_active
0,CUST001,john smith,john.smith@email.com,555-1234,2024-01-15,$45000,US,32.0,Yes
1,CUST002,JANE DOE,jane.doe@example.org,(555) 123-4567,01/15/2024,$78000,USA,28.0,1
2,CUST003,John Smith,john.smith@email.com,5551234567,January 15 2024,$45000,United States,32.0,TRUE
3,CUST004,bob johnson,bob@invalid@email.com,555-0987,2024-02-20,$120000,US,45.0,No
4,CUST005,Alice Williams,alice.w@domain.com,(555) 098-7654,02/20/2024,98500,US,29.0,0


## 원본 데이터 문제점 확인

In [4]:
print('=== 데이터 타입 및 결측값 ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    '결측값': df.isnull().sum(),
    '결측률(%)': (df.isnull().sum() / len(df) * 100).round(1),
    '고유값 수': df.nunique()
})
print(info.to_string())

=== 데이터 타입 및 결측값 ===
                  dtype  결측값  결측률(%)  고유값 수
CustomerID       object    0     0.0    100
customer_name    object    0     0.0    100
Email            object    8     8.0     57
Phone            object    1     1.0     33
signup_date      object    5     5.0     88
Annual_Revenue   object    7     7.0     88
Country          object    0     0.0      4
Age             float64    1     1.0     32
is_active        object    0     0.0     12


## 1. customer_name → Title Case

In [5]:
print('[전] 이름 샘플:', df['customer_name'].head(6).tolist())
df['customer_name'] = df['customer_name'].str.strip().str.title()
print('[후] 이름 샘플:', df['customer_name'].head(6).tolist())

[전] 이름 샘플: ['john smith', 'JANE DOE', 'John Smith', 'bob johnson', 'Alice Williams', 'ALICE WILLIAMS']
[후] 이름 샘플: ['John Smith', 'Jane Doe', 'John Smith', 'Bob Johnson', 'Alice Williams', 'Alice Williams']


## 2. Email 유효성 검사 + email_valid 플래그

In [6]:
_EMAIL_RE = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')

def is_valid_email(email):
    if pd.isna(email) or str(email).strip() == '':
        return False
    e = str(email).strip()
    return (e.count('@') == 1) and ('..' not in e) and bool(_EMAIL_RE.match(e))

df['email_valid'] = df['Email'].map(is_valid_email)

print('=== 무효 이메일 목록 ===')
invalid = df[df['email_valid'] == False][['CustomerID', 'customer_name', 'Email']].dropna(subset=['Email'])
print(invalid.to_string(index=False))
print(f'\n유효: {df["email_valid"].sum()}건 / 무효: {(~df["email_valid"]).sum()}건')

=== 무효 이메일 목록 ===
CustomerID customer_name                     Email
   CUST004   Bob Johnson     bob@invalid@email.com
   CUST015 George Martin       george@invalid..com
   CUST031  Oscar Martin           oscar@invalid..
   CUST057 Benjamin Reed   benjamin.r@invalid..com
   CUST065    Felix Hart felix@invalid@email.co.uk
   CUST073    James Park       james@invalid...com
   CUST085 Peter Johnson        peter@invalid..com
   CUST093   Tanya Leone   tanya@invalid@email.net
   CUST099  Walter White       walter@invalid..com

유효: 83건 / 무효: 17건


## 3. Phone → (XXX) XXX-XXXX

In [7]:
def standardize_phone(phone):
    if pd.isna(phone) or str(phone).strip() == '':
        return np.nan
    digits = re.sub(r'\D', '', str(phone))
    if len(digits) == 10:
        return f'({digits[:3]}) {digits[3:6]}-{digits[6:]}'
    return np.nan  # 7자리 등 불완전 번호는 NaN

before_sample = df['Phone'].head(6).tolist()
df['Phone'] = df['Phone'].map(standardize_phone)
after_sample = df['Phone'].head(6).tolist()

print('[전]', before_sample)
print('[후]', after_sample)
print(f'\n변환 불가(NaN): {df["Phone"].isna().sum()}건')

[전] ['555-1234', '(555) 123-4567', '5551234567', '555-0987', '(555) 098-7654', '5550987654']
[후] [nan, '(555) 123-4567', '(555) 123-4567', nan, '(555) 098-7654', '(555) 098-7654']

변환 불가(NaN): 34건


## 4. signup_date → YYYY-MM-DD

In [8]:
def parse_date(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    s = str(val).strip().strip('()')  # 괄호 제거: "(May 05 2024)" 등
    try:
        return dateparser.parse(s, dayfirst=False).strftime('%Y-%m-%d')
    except Exception:
        return np.nan

# 다양한 날짜 형식 샘플 확인
date_formats = df_orig['signup_date'].dropna().unique()[:8]
print('[전] 날짜 형식 샘플:', date_formats.tolist())

df['signup_date'] = df['signup_date'].map(parse_date)
print('[후] 샘플:', df['signup_date'].dropna().head(8).tolist())

[전] 날짜 형식 샘플: ['2024-01-15', '01/15/2024', 'January 15 2024', '2024-02-20', '02/20/2024', '2024-03-10', '03/10/2024', 'March 10 2024']
[후] 샘플: ['2024-01-15', '2024-01-15', '2024-01-15', '2024-02-20', '2024-02-20', '2024-02-20', '2024-03-10', '2024-03-10']


## 5. Annual_Revenue → float

In [9]:
def clean_revenue(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    s = re.sub(r'[\$,\s]', '', str(val))  # $$105000 같은 이중 기호도 처리
    try:
        return float(s)
    except Exception:
        return np.nan

before_sample = df_orig['Annual_Revenue'].dropna().head(6).tolist()
df['Annual_Revenue'] = df['Annual_Revenue'].map(clean_revenue)

print('[전]', before_sample)
print('[후]', df['Annual_Revenue'].dropna().head(6).tolist())
print(f'\ndtype: {df["Annual_Revenue"].dtype}')

[전] ['$45000', '$78000', '$45000', '$120000', '98500', '98500']
[후] [45000.0, 78000.0, 45000.0, 120000.0, 98500.0, 98500.0]

dtype: float64


## 6. Country → 전체 국가명

In [10]:
COUNTRY_MAP = {
    'US': 'United States', 'USA': 'United States', 'UNITED STATES': 'United States',
    'UK': 'United Kingdom', 'UNITED KINGDOM': 'United Kingdom',
}

print('[전] 고유값:', df_orig['Country'].str.strip().unique().tolist())
df['Country'] = df['Country'].str.strip().str.upper().map(COUNTRY_MAP)
print('[후] 고유값:', df['Country'].unique().tolist())

[전] 고유값: ['US', 'USA', 'United States', 'UK']
[후] 고유값: ['United States', 'United Kingdom']


## 7. Age → 정수, 불가능한 값 제거

In [11]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

invalid_ages = df[~df['Age'].isna() & ((df['Age'] < 0) | (df['Age'] > 120))][['CustomerID', 'customer_name', 'Age']]
print('=== 제거된 불가능한 Age 값 ===')
print(invalid_ages.to_string(index=False))

df.loc[(df['Age'] < 0) | (df['Age'] > 120), 'Age'] = np.nan
df['Age'] = df['Age'].astype('Int64')  # nullable integer (NaN 허용)

print(f'\nAge 범위: {df["Age"].min()} ~ {df["Age"].max()}')

=== 제거된 불가능한 Age 값 ===
CustomerID customer_name   Age
   CUST010  Diana Prince  -5.0
   CUST015 George Martin 150.0

Age 범위: 22 ~ 58


## 8. is_active → True / False

In [12]:
_TRUE  = {'yes', '1', 'true'}
_FALSE = {'no',  '0', 'false'}

def to_bool(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in _TRUE:  return True
    if s in _FALSE: return False
    return np.nan

print('[전] 고유값:', df_orig['is_active'].unique().tolist())
df['is_active'] = df['is_active'].map(to_bool)
print('[후] 분포:')
print(df['is_active'].value_counts(dropna=False).to_string())

[전] 고유값: ['Yes', '1', 'TRUE', 'No', '0', 'FALSE', 'True', 'YES', 'true', 'false', 'yes', 'no']
[후] 분포:
is_active
True     59
False    41


## 9. 중복 행 제거

In [13]:
# 완성도 높은 행 우선 유지 (non-null 개수 기준 정렬)
df['_score'] = df.notna().sum(axis=1)
df = df.sort_values('_score', ascending=False)

# ① 유효 이메일 기준 중복 제거
valid_mask = df['email_valid']
deduped_by_email = df[valid_mask].drop_duplicates(subset='Email', keep='first')
remaining        = df[~valid_mask]

# ② 유효 이메일 없는 행은 (이름 + 나이 + 매출) 기준 추가 중복 제거
df_merged = pd.concat([deduped_by_email, remaining])
df_clean  = df_merged.drop_duplicates(subset=['customer_name', 'Age', 'Annual_Revenue'], keep='first')
df_clean  = df_clean.drop(columns=['_score'])
df_clean  = df_clean.sort_values('CustomerID').reset_index(drop=True)

print(f'중복 제거: {len(df)} → {len(df_clean)}행 ({len(df) - len(df_clean)}건 제거)')

중복 제거: 100 → 57행 (43건 제거)


## 10. 전/후 비교

In [14]:
print('=' * 55)
print('데이터 정제 전/후 비교 요약')
print('=' * 55)
print(f'\n행 수  : {df_orig.shape[0]}행  →  {df_clean.shape[0]}행')
print(f'열 수  : {df_orig.shape[1]}열  →  {df_clean.shape[1]}열 (email_valid 추가)')

print('\n[결측값 변화]')
orig_cols = df_orig.columns.tolist()
miss_df = pd.DataFrame({
    '원본 결측': df_orig[orig_cols].isnull().sum(),
    '정제 후 결측': df_clean.reindex(columns=orig_cols).isnull().sum()
})
miss_df['변화'] = miss_df['정제 후 결측'] - miss_df['원본 결측']
print(miss_df[miss_df[['원본 결측', '정제 후 결측']].sum(axis=1) > 0].to_string())

print('\n[이메일 유효성]')
print(df_clean['email_valid'].value_counts(dropna=False).to_string())

print('\n[Country 통일 결과]')
print(df_clean['Country'].value_counts(dropna=False).to_string())

print('\n[is_active 분포]')
print(df_clean['is_active'].value_counts(dropna=False).to_string())

데이터 정제 전/후 비교 요약

행 수  : 100행  →  57행
열 수  : 9열  →  10열 (email_valid 추가)

[결측값 변화]
                원본 결측  정제 후 결측  변화
Email               8        3  -5
Phone               1       10   9
signup_date         5        1  -4
Annual_Revenue      7        6  -1
Age                 1        2   1

[이메일 유효성]
email_valid
True     48
False     9

[Country 통일 결과]
Country
United States     56
United Kingdom     1

[is_active 분포]
is_active
True     33
False    24


In [15]:
print('=== 정제된 데이터 미리보기 (상위 10행) ===')
df_clean.head(10)

=== 정제된 데이터 미리보기 (상위 10행) ===


,CustomerID,customer_name,Email,Phone,signup_date,Annual_Revenue,Country,Age,is_active,email_valid
0,CUST002,Jane Doe,jane.doe@example.org,(555) 123-4567,2024-01-15,78000.0,United States,28,True,True
1,CUST003,John Smith,john.smith@email.com,(555) 123-4567,2024-01-15,45000.0,United States,32,True,True
2,CUST004,Bob Johnson,bob@invalid@email.com,NaN,2024-02-20,120000.0,United States,45,False,False
3,CUST006,Alice Williams,alice.w@domain.com,(555) 098-7654,2024-02-20,98500.0,United States,29,False,True
4,CUST008,Charlie Brown,charlie.b@mail.net,(555) 567-8901,2024-03-10,67000.0,United States,35,True,True
5,CUST009,Diana Prince,diana@email.co.uk,(555) 567-8901,2024-03-10,156000.0,United Kingdom,41,True,True
6,CUST012,Evan Hunt,evan@example.com,(555) 234-5678,2024-04-05,89500.0,United States,27,True,True
7,CUST014,Fiona Green,fiona.green@email.net,(555) 345-6789,2024-04-15,52000.0,United States,33,False,True
8,CUST015,George Martin,george@invalid..com,(555) 345-6789,2024-04-15,NaN,United States,<NA>,True,False
9,CUST016,George Martin,NaN,NaN,2024-05-01,195000.0,United States,58,False,False


## 저장

In [16]:
df_clean.to_csv('cleaned_customer_data.csv', index=False)
print(f'cleaned_customer_data.csv 저장 완료')
print(f'최종 크기: {df_clean.shape[0]}행 × {df_clean.shape[1]}열')

cleaned_customer_data.csv 저장 완료
최종 크기: 57행 × 10열
